In [1]:
#!pip install -q bertopic sentence-transformers umap-learn hdbscan scikit-learn scikit-fuzzy openpyxl plotly


In [2]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, hamming_loss, accuracy_score

d:\semester-7\quran hadis mining\index_quran\index_kisah_nabi\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("Alquran_Indonesian_Translate.csv")
df["no_surah"] = df["no_surah"].astype(int)
df["no_ayat"] = df["no_ayat"].astype(int)
df = df.sort_values(["no_surah", "no_ayat"]).reset_index(drop=True)
df.head()

,no_surah,no_ayat,nama_surah,teks_terjemahan
0,1,1,Al-Fatihah,Dengan menyebut nama Allah Yang Maha Pemurah l...
1,1,2,Al-Fatihah,"Segala puji bagi Allah, Tuhan semesta alam."
2,1,3,Al-Fatihah,Maha Pemurah lagi Maha Penyayang.
3,1,4,Al-Fatihah,Yang menguasai di Hari Pembalasan.
4,1,5,Al-Fatihah,"Hanya Engkaulah yang kami sembah, dan hanya ke..."


In [4]:
NABI_KEYWORDS = {
    "Adam": ["adam"],
    "Idris": ["idris"],
    "Nuh": ["nuh"],
    "Hud": ["hud", "kaum 'ad", "kaum ad"],
    "Shalih": ["shalih", "tsamud"],
    "Ibrahim": ["ibrahim", "namrudz", "namrud"],
    "Luth": ["luth"],
    "Ismail": ["ismail"],
    "Ishaq": ["ishaq"],
    "Ya'qub": ["ya'qub", "yaqub", "yakub", "(?<!bani )israil"],
    "Yusuf": ["yusuf", "zulaikha"],
    "Syu'aib": ["syu'aib", "syuaib", "madyan"],
    "Ayyub": ["ayyub"],
    "Musa": ["musa", "fir'aun", "firaun", "qarun", "haman", "samiri", "bani israil", "tursina", "thursina"],
    "Harun": ["harun"],
    "Dzulkifli": ["dzulkifli"],
    "Dawud": ["dawud", "daud"],
    "Sulaiman": ["sulaiman", "balqis"],
    "Ilyas": ["ilyas"],
    "Ilyasa": ["ilyasa"],
    "Yunus": ["yunus", "dzun nun", "dzunnun"],
    "Zakariya": ["zakariya", "zakaria"],
    "Yahya": ["yahya"],
    "Isa": ["isa", "maryam"],
    "Muhammad": ["muhammad", "ahmad"],
}

NABI_PATTERNS = {}
for nabi, patterns in NABI_KEYWORDS.items():
    regex_parts = []
    for p in patterns:
        if "(?<!" in p:
            # Jika pola adalah negative lookbehind, biarkan sintaks aslinya (jangan di-escape)
            regex_parts.append(r"\b{}\b".format(p))
        else:
            # Jika teks biasa, escape karakter anehnya (seperti tanda petik pada ya'qub)
            regex_parts.append(r"\b{}\b".format(re.escape(p)))
            
    gabungan_pola = "|".join(regex_parts)
    NABI_PATTERNS[nabi] = re.compile(gabungan_pola, re.IGNORECASE)

# Matriks boolean (ayat x nabi): satu kolom per nabi, dihitung tanpa .apply per baris
texts = df["teks_terjemahan"].astype(str)
hits = pd.DataFrame({nabi: texts.str.contains(pat, regex=True) for nabi, pat in NABI_PATTERNS.items()})

# Ubah matriks boolean menjadi list nama nabi per ayat (urutan sesuai NABI_KEYWORDS)
nabi_names = hits.columns.to_numpy()
df["nabi_terdeteksi"] = [nabi_names[row].tolist() for row in hits.to_numpy()]
df["is_anchor"] = hits.any(axis=1)
anchor_indices = df.index[df["is_anchor"]].to_numpy()

print(f"Jumlah ayat jangkar: {len(anchor_indices)}")
print(hits.sum().sort_values(ascending=False))

Jumlah ayat jangkar: 756
Musa         245
Muhammad     127
Ibrahim      100
Nuh           67
Yusuf         57
Isa           55
Luth          42
Adam          28
Shalih        25
Harun         24
Sulaiman      24
Dawud         18
Zakariya      11
Hud            9
Ismail         9
Ishaq          8
Yunus          5
Yahya          5
Ilyas          4
Ayyub          4
Syu'aib        3
Idris          2
Ya'qub         2
Dzulkifli      1
Ilyasa         1
dtype: int64


In [5]:
WINDOW = 2

def build_context_block(df, anchor_idx, window=WINDOW):
    # df sudah terurut (no_surah, no_ayat) dengan index 0..n-1, jadi jendela ayat cukup diambil lewat slice
    anchor = df.iloc[anchor_idx]
    block = df.iloc[anchor_idx : anchor_idx + window + 1]

    # Simpan hanya ayat pada surah yang sama dan dalam rentang [anchor, anchor + window]
    block = block[
        (block["no_surah"] == anchor["no_surah"])
        & (block["no_ayat"] <= anchor["no_ayat"] + window)
    ]

    return {
        "no_surah": int(anchor["no_surah"]),
        "nama_surah": anchor["nama_surah"],
        "anchor_ayat": int(anchor["no_ayat"]),
        "ayat_range": block["no_ayat"].astype(int).tolist(),
        "teks_ayat": str(anchor["teks_terjemahan"]),
        "teks_gabungan": " ".join(block["teks_terjemahan"].astype(str)),
        "nabi_terdeteksi": sorted({n for labels in block["nabi_terdeteksi"] for n in labels}),
    }

context_df = (
    pd.DataFrame([build_context_block(df, i) for i in anchor_indices])
    .drop_duplicates(subset="teks_gabungan")
    .reset_index(drop=True)
)
print(f"Jumlah blok konteks unik: {len(context_df)}")

docs = context_df["teks_gabungan"].tolist()

Jumlah blok konteks unik: 756


In [6]:
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 24/24 [01:34<00:00,  3.95s/it]


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

daftar_stop_words = [
    "dan", "yang", "di", "ke", "dari", "itu", "ini", "mereka", "kami", "kamu", "aku",
    "dia", "ia", "orang", "kepada", "untuk", "dengan", "maka", "tidak", "akan", "telah",
    "sesungguhnya", "bahwa", "adalah", "pada", "dalam", "atas", "apabila", "ketika", "lalu",
    "nya", "tiap"
]

vectorizer_model = CountVectorizer(stop_words=daftar_stop_words)

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=6,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)
topic_info = topic_model.get_topic_info()
print(topic_info)

def get_topic_label(topic_id, model, top_n_words=5):
    if topic_id == -1:
        return "Outlier / Tidak Terklasifikasi"
    return ", ".join(w for w, _ in model.get_topic(topic_id)[:top_n_words])

# Hitung label sekali per topik unik, lalu petakan ke seluruh dokumen
topic_labels = {t: get_topic_label(t, topic_model) for t in set(topics)}
context_df["cluster_id"] = topics
context_df["cluster_label"] = context_df["cluster_id"].map(topic_labels)

2026-09-29 23:17:30,948 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-29 23:17:45,665 - BERTopic - Dimensionality - Completed ✓
2026-09-29 23:17:45,666 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-29 23:17:45,732 - BERTopic - Cluster - Completed ✓
2026-09-29 23:17:45,743 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-29 23:17:45,814 - BERTopic - Representation - Completed ✓


    Topic  Count                                         Name  \
0      -1    110                       -1_yang_dan_allah_kamu   
1       0    112                 0_muhammad_orang_mereka_yang   
2       1     60                         1_nuh_orang_dan_kami   
3       2     54                   2_yusuf_wanita_dia_berkata   
4       3     26                      3_luth_azab_batu_utusan   
5       4     26                  4_adam_berfirman_iblis_kamu   
6       5     26                 5_sulaiman_daud_burung_semut   
7       6     22                      6_mereka_kami_dan_telah   
8       7     22             7_ibrahim_berhala_patung_berkata   
9       8     21                      8_fir_aun_mukjizat_musa   
10      9     21              9_ibrahim_jadikanlah_dan_kepada   
11     10     18                     10_masing_kami_dan_hamba   
12     11     17                     11_musa_engkau_dan_telah   
13     12     16                      12_shaleh_hud_kamu_kaum   
14     13     16         

In [8]:
from sklearn.metrics import silhouette_score

reduced_embeddings = topic_model.umap_model.embedding_
topics_arr = np.array(topics)
mask = topics_arr != -1
outlier_ratio = 1 - mask.mean()

if mask.sum() > 1 and len(set(topics_arr[mask])) > 1:
    sil_score = silhouette_score(reduced_embeddings[mask], topics_arr[mask])
else:
    sil_score = float("nan")

n_topics_valid = topic_info[topic_info["Topic"] != -1].shape[0]
print(f"Jumlah topik (tanpa outlier): {n_topics_valid}")
print(f"Proporsi ayat outlier: {outlier_ratio:.2%}")
print(f"Silhouette Score (tanpa outlier): {sil_score:.4f}")

Jumlah topik (tanpa outlier): 35
Proporsi ayat outlier: 14.55%
Silhouette Score (tanpa outlier): 0.5973


In [9]:
hierarchical_topics = topic_model.hierarchical_topics(docs)

fig_hierarchy = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig_hierarchy.show()

fig_barchart = topic_model.visualize_barchart(top_n_topics=min(15, len(topic_info)))
fig_barchart.show()

  0%|          | 0/34 [00:00<?, ?it/s]d:\semester-7\quran hadis mining\index_quran\index_kisah_nabi\.venv\lib\site-packages\bertopic\_bertopic.py:1160: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selection.Topic = 0
d:\semester-7\quran hadis mining\index_quran\index_kisah_nabi\.venv\lib\site-packages\bertopic\_bertopic.py:1160: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selection.Topic = 0
d:\semester-7\quran hadis mining\index_quran\index_kisah_nabi\.venv\lib\site-packages\bertopic\_bertopic.py:1160: 

In [10]:
exploded = context_df.explode("nabi_terdeteksi")
crosstab = pd.crosstab(exploded["cluster_label"], exploded["nabi_terdeteksi"])
crosstab

nabi_terdeteksi,Adam,Ayyub,Dawud,Dzulkifli,Harun,Hud,Ibrahim,Idris,Ilyas,Ilyasa,...,Musa,Nuh,Shalih,Sulaiman,Syu'aib,Ya'qub,Yahya,Yunus,Yusuf,Zakariya
cluster_label,,,,,,,,,,,,,,,,,,,,,
Outlier / Tidak Terklasifikasi,1,0,2,0,7,0,21,0,0,0,...,65,3,4,3,0,1,2,0,0,1
"adam, berfirman, iblis, kamu, surga",26,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"akan, sekalian, kamu, ini, lebih",0,0,0,0,5,0,0,0,0,0,...,5,0,0,0,0,0,0,0,1,0
"api, tempat, musa, keluarganya, diserulah",0,0,0,0,0,0,0,0,0,0,...,6,0,0,0,0,0,0,0,0,0
"aun, fir, pintu, aku, benar",0,0,0,0,0,0,0,0,0,0,...,6,0,0,0,0,0,0,0,0,0
"fir, aun, mukjizat, musa, sihir",0,0,0,0,3,0,0,0,0,0,...,21,0,1,0,0,0,0,0,0,0
"ibrahim, berhala, patung, berkata, kamu",0,0,0,0,1,0,22,0,0,0,...,1,1,0,0,0,0,0,0,0,0
"ibrahim, jadikanlah, dan, kepada, ya",0,0,0,0,0,0,21,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"isa, hidangan, allah, kami, yang",0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [11]:
import skfuzzy as fuzz

n_clusters_fuzzy = n_topics_valid

cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    reduced_embeddings.T, c=n_clusters_fuzzy, m=2, error=0.005, maxiter=1000, seed=42,
)
print(f"Fuzzy partition coefficient (FPC): {fpc:.4f}")

# Untuk tiap klaster fuzzy: ambil top-k dokumen dengan keanggotaan tertinggi, lalu ambil topik BERTopic terbanyak
top_k = 5
topics_arr = np.asarray(topics)
top_docs = np.argsort(-u, axis=1)[:, :top_k]
auto_labels = {
    c_idx: get_topic_label(int(Counter(topics_arr[idx]).most_common(1)[0][0]), topic_model)
    for c_idx, idx in enumerate(top_docs)
}
for c_idx, label in auto_labels.items():
    print(f"Bab Index {c_idx + 1}: {label}")

Fuzzy partition coefficient (FPC): 0.4608
Bab Index 1: isa, hidangan, allah, kami, yang
Bab Index 2: Outlier / Tidak Terklasifikasi
Bab Index 3: ibrahim, berhala, patung, berkata, kamu
Bab Index 4: muhammad, orang, mereka, yang, dan
Bab Index 5: luth, azab, batu, utusan, pengikut
Bab Index 6: yusuf, wanita, dia, berkata, sesungguhnya
Bab Index 7: ibrahim, jadikanlah, dan, kepada, ya
Bab Index 8: sulaiman, daud, burung, semut, dan
Bab Index 9: masing, kami, dan, hamba, baik
Bab Index 10: rasul, kaum, mendustakan, aikah, tsamud
Bab Index 11: musa, ia, perempuan, aib, syu
Bab Index 12: musa, engkau, dan, telah, yang
Bab Index 13: muhammad, orang, mereka, yang, dan
Bab Index 14: musa, engkau, dan, telah, yang
Bab Index 15: shaleh, hud, kamu, kaum, yang
Bab Index 16: rasul, allah, mereka, nya, dan
Bab Index 17: mereka, kami, dan, telah, yang
Bab Index 18: muhammad, orang, mereka, yang, dan
Bab Index 19: isa, hidangan, allah, kami, yang
Bab Index 20: muhammad, orang, mereka, yang, dan
Bab In

In [12]:
# Pemetaan manual: Key adalah c_idx (0 sampai 34), Value adalah Nama Bab Indeks yang rapi
manual_mapping = {
    0: "Kisah Nabi Isa (Mukjizat Hidangan)",             # Bab 1
    1: "Outlier / Tidak Terklasifikasi",                 # Bab 2
    2: "Kisah Nabi Ibrahim (Menghancurkan Berhala)",     # Bab 3
    3: "Kisah Nabi Muhammad dan Umatnya",                # Bab 4
    4: "Kisah Nabi Luth (Azab Kaum Sodom)",              # Bab 5
    5: "Kisah Nabi Yusuf dan Zulaikha",                  # Bab 6
    6: "Kisah Nabi Ibrahim (Doa dan Keturunan)",         # Bab 7
    7: "Kisah Nabi Sulaiman dan Daud",                   # Bab 8
    8: "Narasi Umum: Balasan bagi Hamba",                # Bab 9
    9: "Kisah Nabi Syu'aib dan Shaleh (Aikah & Tsamud)", # Bab 10
    10: "Kisah Nabi Musa (Pelarian ke Madyan)",          # Bab 11
    11: "Kisah Nabi Musa (Dialog Tursina)",              # Bab 12
    12: "Kisah Nabi Muhammad dan Umatnya",               # Bab 13
    13: "Kisah Nabi Musa (Dialog Tursina)",              # Bab 14
    14: "Kisah Nabi Hud dan Shaleh",                     # Bab 15
    15: "Narasi Umum: Pengutusan Rasul",                 # Bab 16
    16: "Narasi Umum: Balasan Umat Terdahulu",           # Bab 17
    17: "Kisah Nabi Muhammad dan Umatnya",               # Bab 18
    18: "Kisah Nabi Isa (Mukjizat Hidangan)",            # Bab 19
    19: "Kisah Nabi Muhammad dan Umatnya",               # Bab 20
    20: "Kisah Nabi Nuh dan Umatnya",                    # Bab 21
    21: "Kisah Nabi Nuh dan Umatnya",                    # Bab 22
    22: "Kisah Nabi Nuh dan Umatnya",                    # Bab 23
    23: "Kisah Nabi Ibrahim (Kedatangan Malaikat)",      # Bab 24
    24: "Kisah Nabi Shaleh (Kaum Tsamud)",               # Bab 25
    25: "Kisah Nabi Musa (Fir'aun & Ahli Sihir)",        # Bab 26
    26: "Kisah Nabi Muhammad dan Umatnya",               # Bab 27
    27: "Kisah Nabi Adam dan Iblis",                     # Bab 28
    28: "Kisah Maryam dan Kelahiran Isa",                # Bab 29
    29: "Kisah Nabi Musa (Mukjizat Tongkat & Ular)",     # Bab 30
    30: "Narasi Umum: Balasan bagi Hamba",               # Bab 31
    31: "Narasi Umum: Pengutusan Rasul",                 # Bab 32
    32: "Narasi Umum: Ujian Keimanan",                   # Bab 33
    33: "Kisah Nabi Luth (Kedatangan Utusan Malaikat)",  # Bab 34
    34: "Kisah Nabi Isa (Mukjizat Hidangan)"             # Bab 35
}

OUTLIER_LABEL = "Outlier / Tidak Terklasifikasi"

# Pemetaan c_idx -> label manual; klaster yang tidak ada di mapping jatuh ke label outlier
bab_index_labels = {c: manual_mapping.get(c, OUTLIER_LABEL) for c in range(n_clusters_fuzzy)}

# Array label (dibersihkan dari prefiks angka "12_" bila ada) agar bisa di-index secara vektor
cluster_label_array = np.array([re.sub(r"^\d+_", "", bab_index_labels[c]) for c in range(n_clusters_fuzzy)])

for c, label in enumerate(cluster_label_array, start=1):
    print(f"Bab Index {c}: {label}")

Bab Index 1: Kisah Nabi Isa (Mukjizat Hidangan)
Bab Index 2: Outlier / Tidak Terklasifikasi
Bab Index 3: Kisah Nabi Ibrahim (Menghancurkan Berhala)
Bab Index 4: Kisah Nabi Muhammad dan Umatnya
Bab Index 5: Kisah Nabi Luth (Azab Kaum Sodom)
Bab Index 6: Kisah Nabi Yusuf dan Zulaikha
Bab Index 7: Kisah Nabi Ibrahim (Doa dan Keturunan)
Bab Index 8: Kisah Nabi Sulaiman dan Daud
Bab Index 9: Narasi Umum: Balasan bagi Hamba
Bab Index 10: Kisah Nabi Syu'aib dan Shaleh (Aikah & Tsamud)
Bab Index 11: Kisah Nabi Musa (Pelarian ke Madyan)
Bab Index 12: Kisah Nabi Musa (Dialog Tursina)
Bab Index 13: Kisah Nabi Muhammad dan Umatnya
Bab Index 14: Kisah Nabi Musa (Dialog Tursina)
Bab Index 15: Kisah Nabi Hud dan Shaleh
Bab Index 16: Narasi Umum: Pengutusan Rasul
Bab Index 17: Narasi Umum: Balasan Umat Terdahulu
Bab Index 18: Kisah Nabi Muhammad dan Umatnya
Bab Index 19: Kisah Nabi Isa (Mukjizat Hidangan)
Bab Index 20: Kisah Nabi Muhammad dan Umatnya
Bab Index 21: Kisah Nabi Nuh dan Umatnya
Bab Index 

In [13]:
FUZZY_THRESHOLD = 0.10
CLASSIFIER_THRESHOLD = 0.3
FALLBACK_NABI = "Nabi Lainnya / Kisah Umum"

def threshold_mask(scores, threshold):
    # scores: matriks (n_item x n_kelas). True bila skor >= threshold
    mask = scores >= threshold
    # Fallback: item tanpa skor yang lolos threshold tetap diberi kelas dengan skor tertinggi
    empty = ~mask.any(axis=1)
    mask[empty, scores[empty].argmax(axis=1)] = True
    return mask

def select_fuzzy_memberships(u, threshold=FUZZY_THRESHOLD):
    # u dari skfuzzy berbentuk (n_klaster x n_dokumen); transpose menjadi (n_dokumen x n_klaster)
    U = u.T
    doc_idx, c_idx = np.nonzero(threshold_mask(U, threshold))
    # Format long: satu baris per pasangan (dokumen, klaster) yang lolos threshold
    return pd.DataFrame({"doc_idx": doc_idx, "c_idx": c_idx, "derajat": U[doc_idx, c_idx]})

fuzzy_long = select_fuzzy_memberships(u)
fuzzy_long["label"] = cluster_label_array[fuzzy_long["c_idx"]]

# ---------- Tabel flat per klaster mesin (setara sel 12 asli) ----------
context_meta = context_df[["no_surah", "nama_surah", "anchor_ayat", "ayat_range", "teks_ayat", "teks_gabungan"]].assign(
    nabi_rule_based=context_df["nabi_terdeteksi"].str.join(", ")
)

fuzzy_df = (
    fuzzy_long.join(context_meta, on="doc_idx")
    .assign(
        bab_index=lambda d: d["c_idx"] + 1,
        bab_index_label=lambda d: d["label"],
        derajat_keanggotaan=lambda d: (d["derajat"] * 100).round(2),
    )
    .sort_values(["no_surah", "anchor_ayat", "derajat_keanggotaan"], ascending=[True, True, False], kind="stable")
    [["no_surah", "nama_surah", "anchor_ayat", "ayat_range", "teks_gabungan",
      "nabi_rule_based", "bab_index", "bab_index_label", "derajat_keanggotaan"]]
    .reset_index(drop=True)
)

bab_counts = fuzzy_df["bab_index"].value_counts().reindex(range(1, n_clusters_fuzzy + 1), fill_value=0)
fuzzy_summary_df = pd.DataFrame({
    "bab_index": bab_counts.index,
    "bab_index_label": cluster_label_array,
    "jumlah_ayat_terhubung": bab_counts.to_numpy(),
})
print(f"Jumlah baris fuzzy (pasangan ayat-bab_index): {len(fuzzy_df)}")

Jumlah baris fuzzy (pasangan ayat-bab_index): 1401


In [14]:
# ---------- Tabel hierarkis: Nabi -> Sub_Kisah -> Surah -> Ayat ----------

# 1) Gabungkan klaster yang bernama sama SEBELUM dibulatkan: derajat mentah dijumlahkan sekali
#    per (dokumen, label), sehingga tidak ada baris ganda untuk label yang sama
def select_label_memberships(u, labels, threshold=FUZZY_THRESHOLD):
    # Jumlahkan keanggotaan klaster bernama sama TERLEBIH DAHULU -> matriks (n_dokumen x n_label)
    L = pd.DataFrame(u).groupby(labels).sum().T
    scores = L.to_numpy()

    # Threshold + fallback diterapkan pada skor gabungan per label
    doc_idx, l_idx = np.nonzero(threshold_mask(scores, threshold))
    return pd.DataFrame({
        "doc_idx": doc_idx,
        "label": L.columns.to_numpy()[l_idx],
        "derajat": scores[doc_idx, l_idx],
    })

fuzzy_by_label = select_label_memberships(u, cluster_label_array)

assert fuzzy_by_label.groupby("doc_idx")["derajat"].sum().le(1 + 1e-9).all()
print(f"Proporsi ayat multi-label (>1 sub-kisah): {(fuzzy_by_label.groupby('doc_idx').size() > 1).mean():.2%}")

# 2) Explode daftar nabi: satu ayat dengan N nabi menjadi N baris (multi-label nabi dipertahankan)
nabi_long = (
    context_df["nabi_terdeteksi"]
    .map(lambda x: x or [FALLBACK_NABI])
    .explode()
    .rename("Bab_Utama (Nabi)")
    .rename_axis("doc_idx")
    .reset_index()
)

# 3) Gabungkan nabi x sub-kisah untuk tiap dokumen, sambil membawa metadata surah/ayat
HIER_COLS = ["Bab_Utama (Nabi)", "Sub_Kisah (Klaster Mesin)", "Probabilitas_Fuzzy (%)", "Surah", "Ayat", "Teks_Ayat", "Teks_Konteks"]

df_rapi = (
    fuzzy_by_label
    .merge(nabi_long, on="doc_idx")
    .join(context_df[["no_surah", "nama_surah", "anchor_ayat", "teks_ayat", "teks_gabungan"]], on="doc_idx")
    .assign(**{"Probabilitas_Fuzzy (%)": lambda d: (d["derajat"] * 100).clip(upper=100).round(2)})
    .rename(columns={
        "label": "Sub_Kisah (Klaster Mesin)",
        "nama_surah": "Surah",
        "anchor_ayat": "Ayat",
        "teks_ayat": "Teks_Ayat",
        "teks_gabungan": "Teks_Konteks",
    })
    # Urut hierarki; Surah diurutkan berdasarkan nomor surah (bukan abjad nama surah)
    .sort_values(["Bab_Utama (Nabi)", "Sub_Kisah (Klaster Mesin)", "no_surah", "Ayat"])
    [HIER_COLS]
    .reset_index(drop=True)
)

# Validasi hierarki: tidak boleh ada duplikasi pada kunci Nabi -> Sub_Kisah -> Surah -> Ayat
assert not df_rapi.duplicated(["Bab_Utama (Nabi)", "Sub_Kisah (Klaster Mesin)", "Surah", "Ayat"]).any()

# 4) Ringkasan: jumlah blok ayat unik per nabi (ayat yang muncul di 2 sub-kisah dihitung sekali)
summary_nabi = (
    df_rapi.drop_duplicates(["Bab_Utama (Nabi)", "Surah", "Ayat"])
    .groupby("Bab_Utama (Nabi)").size()
    .sort_values(ascending=False)
    .rename("Jumlah_Blok_Ayat")
    .reset_index()
)

# 5) Ekspor
export_hier_path = "Indeks_Kisah_25_Nabi.xlsx"
with pd.ExcelWriter(export_hier_path) as writer:
    df_rapi.to_excel(writer, sheet_name="Indeks_Lengkap", index=False)
    summary_nabi.to_excel(writer, sheet_name="Ringkasan_Per_Nabi", index=False)
print(f"Tersimpan di: {export_hier_path}")

Proporsi ayat multi-label (>1 sub-kisah): 37.43%
Tersimpan di: Indeks_Kisah_25_Nabi.xlsx


In [15]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(context_df["nabi_terdeteksi"])
X = np.asarray(embeddings)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = OneVsRestClassifier(LogisticRegression(max_iter=1000))
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba_test = clf.predict_proba(X_test)

print(classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0))
print(f"Hamming Loss: {hamming_loss(y_test, y_pred):.4f}")
print(f"Exact Match Ratio: {accuracy_score(y_test, y_pred):.4f}")

              precision    recall  f1-score   support

        Adam       1.00      0.25      0.40         8
       Ayyub       0.00      0.00      0.00         2
       Dawud       0.00      0.00      0.00         5
   Dzulkifli       0.00      0.00      0.00         1
       Harun       0.00      0.00      0.00         7
         Hud       0.00      0.00      0.00         0
     Ibrahim       1.00      0.82      0.90        33
       Idris       0.00      0.00      0.00         1
       Ilyas       0.00      0.00      0.00         2
      Ilyasa       0.00      0.00      0.00         0
         Isa       1.00      0.33      0.50        15
       Ishaq       0.00      0.00      0.00         1
      Ismail       0.00      0.00      0.00         3
        Luth       1.00      0.73      0.84        11
    Muhammad       1.00      0.90      0.95        31
        Musa       0.98      0.80      0.88        50
         Nuh       1.00      0.71      0.83        17
      Shalih       1.00    

In [16]:
from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score, jaccard_score,
    label_ranking_average_precision_score, label_ranking_loss, coverage_error,
    multilabel_confusion_matrix,
)
import plotly.express as px

# Rata-rata micro / macro / weighted dalam satu loop
metrics_summary = pd.DataFrame([
    dict(zip(["average", "precision", "recall", "f1_score"],
             [avg, *precision_recall_fscore_support(y_test, y_pred, average=avg, zero_division=0)[:3]]))
    for avg in ("micro", "macro", "weighted")
])
print(metrics_summary)

jaccard_samples = jaccard_score(y_test, y_pred, average="samples", zero_division=0)
print(f"Jaccard Score (samples): {jaccard_samples:.4f}")

# ROC-AUC bisa gagal bila ada label yang hanya punya satu kelas di data uji
try:
    roc_auc_micro = roc_auc_score(y_test, y_proba_test, average="micro")
    roc_auc_macro = roc_auc_score(y_test, y_proba_test, average="macro")
    print(f"ROC-AUC (micro): {roc_auc_micro:.4f}\nROC-AUC (macro): {roc_auc_macro:.4f}")
except ValueError as e:
    roc_auc_micro = roc_auc_macro = float("nan")
    print(f"ROC-AUC tidak dapat dihitung: {e}")

lrap = label_ranking_average_precision_score(y_test, y_proba_test)
lrl = label_ranking_loss(y_test, y_proba_test)
cov_err = coverage_error(y_test, y_proba_test)
print(f"Label Ranking Average Precision: {lrap:.4f}\nLabel Ranking Loss: {lrl:.4f}\nCoverage Error: {cov_err:.4f}")

# Metrik per nabi
prec, rec, f1, support = precision_recall_fscore_support(y_test, y_pred, average=None, zero_division=0)
per_label_df = (
    pd.DataFrame({"nabi": mlb.classes_, "precision": prec, "recall": rec, "f1_score": f1, "support": support})
    .sort_values("f1_score", ascending=False)
)
px.bar(per_label_df, x="f1_score", y="nabi", orientation="h", title="F1-Score per Nabi (Data Uji)").show()

# Confusion matrix per nabi: reshape (k,2,2) -> (k,4) berurutan tn, fp, fn, tp
confusion_df = pd.DataFrame(
    multilabel_confusion_matrix(y_test, y_pred).reshape(len(mlb.classes_), 4),
    columns=["true_negative", "false_positive", "false_negative", "true_positive"],
).assign(nabi=mlb.classes_)[["nabi", "true_negative", "false_positive", "false_negative", "true_positive"]]
print(confusion_df)

    average  precision    recall  f1_score
0     micro   0.992647  0.622120  0.764873
1     macro   0.439024  0.242849  0.293616
2  weighted   0.869956  0.622120  0.704551
Jaccard Score (samples): 0.7586
ROC-AUC (micro): 0.9643
ROC-AUC (macro): nan
Label Ranking Average Precision: 0.9416
Label Ranking Loss: 0.0222
Coverage Error: 2.2039


         nabi  true_negative  false_positive  false_negative  true_positive
0        Adam            144               0               6              2
1       Ayyub            150               0               2              0
2       Dawud            147               0               5              0
3   Dzulkifli            151               0               1              0
4       Harun            145               0               7              0
5         Hud            152               0               0              0
6     Ibrahim            119               0               6             27
7       Idris            151               0               1              0
8       Ilyas            150               0               2              0
9      Ilyasa            152               0               0              0
10        Isa            137               0              10              5
11      Ishaq            151               0               1              0
12     Ismai

In [17]:
# Latih ulang pada seluruh data, lalu prediksi probabilitas tiap nabi untuk tiap blok konteks
clf_final = OneVsRestClassifier(LogisticRegression(max_iter=1000))
clf_final.fit(X, y)
proba = clf_final.predict_proba(X)

# Multi-label berbasis threshold dengan fallback (fungsi yang sama dengan fuzzy)
doc_idx, cls_idx = np.nonzero(threshold_mask(proba, CLASSIFIER_THRESHOLD))
pred_long = pd.DataFrame({
    "doc_idx": doc_idx,
    "nabi_prediksi_model": mlb.classes_[cls_idx],
    "probabilitas_model": (proba[doc_idx, cls_idx] * 100).round(2),
})

# Tambahkan "teks_ayat" di antara "ayat_range" dan "teks_gabungan"
EXPORT_COLS = ["no_surah", "nama_surah", "anchor_ayat", "ayat_range", "teks_ayat", "teks_gabungan", "nabi_rule_based",
               "cluster_id", "cluster_label", "nabi_prediksi_model", "probabilitas_model", "sesuai_rule_based"]

export_df = pred_long.join(
    context_df[["no_surah", "nama_surah", "anchor_ayat", "ayat_range", "teks_ayat", "teks_gabungan",
                "cluster_id", "cluster_label", "nabi_terdeteksi"]],
    on="doc_idx",
)
export_df["nabi_rule_based"] = export_df["nabi_terdeteksi"].str.join(", ")
export_df["sesuai_rule_based"] = [n in labels for n, labels in zip(export_df["nabi_prediksi_model"], export_df["nabi_terdeteksi"])]
export_df = (
    export_df.sort_values(["no_surah", "anchor_ayat", "probabilitas_model"], ascending=[True, True, False], kind="stable")
    [EXPORT_COLS].reset_index(drop=True)
)

report_df = pd.DataFrame(
    classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0, output_dict=True)
).T

cluster_eval_df = pd.DataFrame([{
    "jumlah_topik": n_topics_valid, "proporsi_outlier": outlier_ratio, "silhouette_score": sil_score,
}])

classification_eval_summary = metrics_summary.assign(
    jaccard_samples=jaccard_samples,
    roc_auc=[roc_auc_micro, roc_auc_macro, float("nan")],
    label_ranking_avg_precision=lrap,
    label_ranking_loss=lrl,
    coverage_error=cov_err,
)

cluster_summary = context_df.groupby(["cluster_id", "cluster_label"]).size().reset_index(name="jumlah_ayat")

# Pastikan juga pada bagian fuzzy_df (jika belum ditambahkan sebelumnya), kolom teks_ayat disertakan:
# context_meta = context_df[["no_surah", "nama_surah", "anchor_ayat", "ayat_range", "teks_ayat", "teks_gabungan"]].assign(...)

export_path = "indeks_kisah_nabi_cluster_klasifikasi.xlsx"
sheets = {
    "Hasil_Klaster_Klasifikasi": (export_df, False),
    "Ringkasan_Klaster": (cluster_summary, False),
    "Evaluasi_Model": (report_df, True),
    "Evaluasi_Klaster": (cluster_eval_df, False),
    "Evaluasi_Ringkasan": (classification_eval_summary, False),
    "Evaluasi_Per_Nabi": (per_label_df, False),
    "Confusion_Per_Nabi": (confusion_df, False),
    "Fuzzy_Multi_Label": (fuzzy_df, False),
    "Ringkasan_Bab_Index_Fuzzy": (fuzzy_summary_df, False),
}
with pd.ExcelWriter(export_path) as writer:
    for sheet_name, (frame, keep_index) in sheets.items():
        frame.to_excel(writer, sheet_name=sheet_name, index=keep_index)
print(f"Hasil tersimpan di: {export_path}")

Hasil tersimpan di: indeks_kisah_nabi_cluster_klasifikasi.xlsx


In [18]:
def top_pairs(scores, names, threshold):
    # Ambil semua kelas yang lolos threshold (fallback ke skor tertinggi), urut menurun
    idx = np.flatnonzero(threshold_mask(scores[None, :], threshold)[0])
    idx = idx[np.argsort(-scores[idx])]
    return [(names[i], round(float(scores[i]) * 100, 2)) for i in idx]

def cari_ayat(input_surah, input_ayat):
    match_idx = df.index[(df["no_surah"] == input_surah) & (df["no_ayat"] == input_ayat)]
    if len(match_idx) == 0:
        print("Ayat tidak ditemukan dalam dataset.")
        return

    block = build_context_block(df, match_idx[0], window=WINDOW)
    query_embedding = embedding_model.encode([block["teks_gabungan"]])

    # Klasifikasi nabi (multi-label berbasis threshold)
    nabi_pred = top_pairs(clf_final.predict_proba(query_embedding)[0], mlb.classes_, CLASSIFIER_THRESHOLD)

    # Keanggotaan fuzzy: proyeksikan ke ruang UMAP, lalu prediksi terhadap centroid yang sudah dilatih
    query_umap = topic_model.umap_model.transform(query_embedding)
    query_u, *_ = fuzz.cluster.cmeans_predict(query_umap.T, cntr, m=2, error=0.005, maxiter=1000, seed=42)
    scores = query_u[:, 0]

    # Terapkan threshold per klaster, lalu jumlahkan klaster yang bernama sama agar label tidak muncul ganda
    mask = threshold_mask(scores[None, :], FUZZY_THRESHOLD)[0]
    bab = pd.Series(scores[mask]).groupby(cluster_label_array[mask]).sum().sort_values(ascending=False)
    bab_pred = [(label, round(float(v) * 100, 2)) for label, v in bab.items()]

    print(f"Surat {block['nama_surah']} ({input_surah}:{input_ayat})")
    print(f"Teks konteks: {block['teks_gabungan']}")
    print(f"Nabi terdeteksi (rule-based): {', '.join(block['nabi_terdeteksi']) or '-'}\n")
    print("Prediksi kisah nabi (model klasifikasi):")
    for nabi, prob in nabi_pred:
        print(f"  {nabi}: {prob}%")
    print("\nKeanggotaan bab index (fuzzy clustering):")
    for label, prob in bab_pred:
        print(f"  {label}: {prob}%")

cari_ayat(int(input("Masukkan nomor surah: ")), int(input("Masukkan nomor ayat: ")))

Surat Al-Baqarah (2:23)
Teks konteks: Dan jika kamu (tetap) dalam keraguan tentang Al Quran yang Kami wahyukan kepada hamba Kami (Muhammad), buatlah satu surat (saja) yang semisal Al Quran itu dan ajaklah penolong-penolongmu selain Allah, jika kamu orang-orang yang benar. Maka jika kamu tidak dapat membuat(nya) -- dan pasti kamu tidak akan dapat membuat(nya), peliharalah dirimu dari neraka yang bahan bakarnya manusia dan batu, yang disediakan bagi orang-orang kafir. Dan sampaikanlah berita gembira kepada mereka yang beriman dan berbuat baik, bahwa bagi mereka disediakan surga-surga yang mengalir sungai-sungai di dalamnya. Setiap mereka diberi rezeki buah-buahan dalam surga-surga itu, mereka mengatakan: "Inilah yang pernah diberikan kepada kami dahulu". Mereka diberi buah-buahan yang serupa dan untuk mereka di dalamnya ada isteri-isteri yang suci dan mereka kekal di dalamnya.
Nabi terdeteksi (rule-based): Muhammad

Prediksi kisah nabi (model klasifikasi):
  Muhammad: 86.15%

Keanggotaan